# Locally Typical Sampling

Demo of **locally typical sampling** from Meister, Pimentel, Wiher & Cotterell, *Locally Typical Sampling*
(citekey `meisterLocallyTypicalSampling2025`, [arXiv:2202.00666](https://arxiv.org/abs/2202.00666)).
Notes: `../llm-inference-notes/notes/papers/meisterLocallyTypicalSampling2025.md`.

**Idea.** High-probability text is often dull or repetitive, yet pure sampling is often incoherent. The paper argues
that humans pick each word so that its information content $-\log p(y \mid y_{<t})$ is close to the model's
*conditional entropy* $H(Y_t \mid y_{<t})$, the expected information content. Locally typical sampling enforces this:
at each step keep the tokens whose information content is closest to $H$, until they cover probability mass $\tau$,
then sample from the renormalised kept set.

* **Part 1** builds intuition for entropy and typicality with a coin and a 4-sided die.
* **Part 2** runs the algorithm, implemented from scratch, on Qwen2.5-1.5B and lets you watch every step.

In [ ]:
import html

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import torch
from IPython.display import clear_output, display

%matplotlib inline

## Part 1: Entropy and typicality

The **information content** (surprisal) of an outcome with probability $p$ is $-\log p$ (nats here). The **entropy**
is its expectation, $H(p) = -\sum_y p(y) \log p(y)$.

### A biased coin

A sequence of $n$ flips has probability $P(\text{seq}) = b^k (1-b)^{n-k}$, where $k$ is the number of heads and $b$ the
bias. Its *per-flip* information content is $-\frac{1}{n} \log P(\text{seq})$.

Sampled sequences concentrate around per-flip information $\approx H$ (the **typical set**). The single most
probable sequence, "always the likelier face", is *not* typical: its per-flip information is lower than $H$, and it
is almost never sampled. Yet that is exactly what likelihood-maximising decoding looks for.

Move the sliders and compare the orange line (most probable sequence) with the histogram of sampled sequences.

In [ ]:
def entropy(p):
    p = np.asarray(p, dtype=float)
    p = p[p > 0]
    return float(-(p * np.log(p)).sum())


BAND = 0.1  # nats: "close to H" for the typical band below


def coin_demo(bias=0.8, n=100):
    rng = np.random.default_rng(0)
    H = entropy([bias, 1 - bias])
    heads = rng.binomial(
        n, bias, size=20_000
    )  # info content depends only on the number of heads
    info = -(heads * np.log(bias) + (n - heads) * np.log(1 - bias)) / n
    best = max(bias, 1 - bias)
    best_info = -np.log(best)

    _fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.hist(info, bins=40, color="#2a6fdb", alpha=0.8, label="sampled sequences")
    ax.axvline(H, color="black", lw=2, label=f"entropy H = {H:.3f}")
    ax.axvline(
        best_info,
        color="#e69f00",
        lw=2,
        ls="--",
        label=f"most probable sequence = {best_info:.3f}",
    )
    ax.set_xlabel("per-flip information content  -log P(seq) / n  (nats)")
    ax.set_ylabel("number of samples")
    ax.legend()
    plt.show()

    in_band = np.mean(np.abs(info - H) < BAND)
    print(f"P(most probable sequence)           = {best**n:.3e}")
    print(
        f"share of samples within H ± {BAND}      = {in_band:.1%}  (the typical band)"
    )


widgets.interact(
    coin_demo,
    bias=widgets.FloatSlider(
        value=0.8, min=0.5, max=0.99, step=0.01, continuous_update=False
    ),
    n=widgets.IntSlider(value=100, min=10, max=500, step=10, continuous_update=False),
);

### The algorithm

At every step, given the next-token distribution $p$:

1. $H = -\sum_y p(y) \log p(y)$
2. deviation of each token: $d(y) = \left| -\log p(y) - H \right|$
3. sort tokens by $d$ ascending and keep the smallest prefix whose cumulative probability reaches $\tau$ (at least
   `min_keep` tokens; ties in $d$ are all kept)
4. renormalise over the kept set and sample from it

Notice that nothing here favours the most probable token: when $p$ is flat-ish the argmax can be dropped, since its
information is *below* $H$.

Below is the from-scratch implementation used for the rest of the notebook. It works on logits, applies the
temperature first, and returns all intermediate quantities so they can be plotted.

In [ ]:
def typical_filter(logits, tau, temperature=1.0, min_keep=1):
    # Locally typical filtering of one next-token distribution (1-D logits).
    logp = torch.log_softmax(logits.float() / temperature, dim=-1)
    p = logp.exp()
    H = (
        -(p * logp).nan_to_num(0.0).sum()
    )  # nan_to_num: 0 * -inf for zero-probability tokens
    dev = (-logp - H).abs()

    dev_sorted, order = torch.sort(dev)  # most typical first
    cum = p[order].cumsum(-1)
    last = min(
        int((cum < tau).sum()), len(cum) - 1
    )  # first sorted position where cum >= tau
    mask = dev <= dev_sorted[last]  # also keeps ties with the boundary token
    mask[order[:min_keep]] = True

    probs_kept = p * mask
    probs_kept = probs_kept / probs_kept.sum()
    return {
        "p": p,
        "logp": logp,
        "H": H,
        "dev": dev,
        "mask": mask,
        "probs_kept": probs_kept,
        "n_keep": int(mask.sum()),
    }

### A 4-sided die

The same function on a tiny "vocabulary". Pick a distribution, then move $\tau$ and the temperature. Bars are sorted by
deviation $|-\log p - H|$, the order in which tokens are admitted to the kept set; green bars are kept.

Try the *skewed* die with $\tau = 0.2$: the most probable face (0.5) is **dropped**, because 0.25 is closer to the
entropy.

In [ ]:
DICE = {
    "uniform": [0.25, 0.25, 0.25, 0.25],
    "skewed": [0.5, 0.25, 0.15, 0.10],
    "peaked": [0.85, 0.08, 0.05, 0.02],
}


def die_demo(dist="skewed", tau=0.5, temperature=1.0):
    p0 = np.array(DICE[dist])
    f = typical_filter(torch.log(torch.tensor(p0)), tau, temperature)
    p, dev, mask, H = f["p"].numpy(), f["dev"].numpy(), f["mask"].numpy(), float(f["H"])
    order = np.argsort(dev, kind="stable")

    fig, ax = plt.subplots(figsize=(7, 3.3))
    ax.bar(
        range(4), p[order], color=["#1b9e77" if mask[i] else "#b0b0b0" for i in order]
    )
    ax.set_xticks(
        range(4), [f"face {i + 1}\np={p[i]:.3f}\nd={dev[i]:.2f}" for i in order]
    )
    ax.set_ylabel("probability")
    ax.set_title(
        f"H = {H:.3f} nats   tau = {tau}   T = {temperature}   kept {f['n_keep']} of 4  (sorted by deviation d)"
    )
    fig.tight_layout()
    plt.show()
    if not mask[np.argmax(p)]:
        print("The most probable face was dropped.")


widgets.interact(
    die_demo,
    dist=list(DICE),
    tau=widgets.FloatSlider(
        value=0.5, min=0.05, max=0.99, step=0.01, continuous_update=False
    ),
    temperature=widgets.FloatSlider(
        value=1.0, min=0.2, max=2.0, step=0.1, continuous_update=False
    ),
);

## Part 2: A real language model

Qwen2.5-1.5B (base model) on Apple-silicon `mps`, about 3 GB in bf16. Generation is stepped manually so that every
step's distribution is recorded.

Edit the constants below; **τ and temperature** are sliders further down.

In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B"
PROMPT = "Once upon a time, in a small village by the sea,"
SEED = 0
MAX_NEW_TOKENS = 40
MIN_TOKENS_TO_KEEP = 1
TOP_N_SHOWN = 30

DEVICE = "mps" if torch.backends.mps.is_available() else "cpu"
DTYPE = torch.bfloat16 if DEVICE == "mps" else torch.float32

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer

tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=DTYPE).to(DEVICE).eval()

### Stepping through generation

`Session` holds the KV cache and the seeded random generator. `advance(tau, temperature)` does one step:

1. take the next-token logits and run `typical_filter`,
2. note the **greedy** token (argmax of the raw logits) for comparison,
3. **sample** from the renormalised kept set,
4. record everything needed to plot the step, then feed the sampled token back into the model.

In [ ]:
def show_token(s):
    return s.replace(" ", "␣").replace("\n", "⏎").replace("\t", "⇥")


class Session:
    def __init__(self):
        self.reset()

    @torch.no_grad()
    def _forward(self, input_ids, cache=None):
        out = model(input_ids.to(DEVICE), past_key_values=cache, use_cache=True)
        self.cache = out.past_key_values
        self.logits = out.logits[0, -1].float().cpu()

    def reset(self):
        self.gen = torch.Generator().manual_seed(SEED)
        self.steps = []
        self.texts = []  # decoded sampled tokens
        self._forward(tok(PROMPT, return_tensors="pt").input_ids)

    def advance(self, tau, temperature):
        f = typical_filter(self.logits, tau, temperature, MIN_TOKENS_TO_KEEP)
        greedy = int(self.logits.argmax())
        sampled = int(torch.multinomial(f["probs_kept"], 1, generator=self.gen))

        top = f["p"].topk(TOP_N_SHOWN).indices.tolist()
        shown = (
            top if sampled in top else top + [sampled]
        )  # sampled token may be a low-probability one
        self.steps.append(
            {
                "ids": shown,
                "labels": [show_token(tok.decode([i])) for i in shown],
                "p": f["p"][shown].numpy(),
                "dev": f["dev"][shown].numpy(),
                "kept": f["mask"][shown].numpy(),
                "H": float(f["H"]),
                "n_keep": f["n_keep"],
                "kept_mass": float(f["p"][f["mask"]].sum()),
                "tau": tau,
                "temperature": temperature,
                "greedy": greedy,
                "sampled": sampled,
                "extra": sampled not in top,
            }
        )
        self.texts.append(tok.decode([sampled]))
        self._forward(torch.tensor([[sampled]]), self.cache)

### Visualisation and controls

For the selected step the chart shows:

* **bars**: token probabilities (top 30 by probability), green = kept by typical sampling, grey = dropped,
* **purple line** (right axis): the deviation $|-\log p - H|$; tokens are admitted from lowest deviation upward,
* ▼ the **greedy** token and ★ the token **typical sampling** actually picked (a separated last bar appears when
  the picked token is outside the top 30).

Buttons: **Regenerate** reruns the whole generation from the prompt with the current τ / temperature (same seed);
**Next token** appends one more step using the current slider values. The step slider scrubs through the
recorded steps.

In [ ]:
KEPT, DROPPED, DEV_LINE, SAMPLED = "#1b9e77", "#b0b0b0", "#7b3294", "#e69f00"


def render_step(s):
    n = len(s["ids"])
    x = np.arange(n)
    fig, ax = plt.subplots(figsize=(12, 4.2))
    ax.bar(x, s["p"], color=[KEPT if k else DROPPED for k in s["kept"]])
    ax.set_xticks(x, s["labels"], rotation=60, ha="right")
    ax.set_ylabel("probability")

    ax2 = ax.twinx()
    ax2.plot(x, s["dev"], "o-", color=DEV_LINE, ms=3, lw=1, label="|-log p - H|")
    ax2.set_ylabel("|-log p - H|", color=DEV_LINE)

    top = s["p"].max()
    gi, si = s["ids"].index(s["greedy"]), s["ids"].index(s["sampled"])
    ax.plot(
        gi - 0.15,
        s["p"][gi] + 0.03 * top,
        "v",
        color="black",
        ms=9,
        label="greedy token",
    )
    ax.plot(
        si + 0.15,
        s["p"][si] + 0.03 * top,
        "*",
        color=SAMPLED,
        mec="black",
        ms=15,
        label="typical sampling pick",
    )
    if s["extra"]:
        ax.axvline(n - 1.5, color="black", ls=":", lw=1)

    handles = [
        plt.Rectangle((0, 0), 1, 1, color=KEPT),
        plt.Rectangle((0, 0), 1, 1, color=DROPPED),
        *ax.get_lines()[:2],
        *ax2.get_lines(),
    ]
    ax.legend(
        handles,
        ["kept", "dropped", "greedy token", "typical sampling pick", "|-log p - H|"],
        loc="upper right",
    )
    ax.set_title(
        f"H = {s['H']:.2f} nats   tau = {s['tau']}   T = {s['temperature']}   "
        f"kept {s['n_keep']} tokens, mass {s['kept_mass']:.2f}"
    )
    fig.tight_layout()
    return fig


def render_text(idx):
    parts = [f"<span style='opacity:.6'>{html.escape(PROMPT)}</span>"]
    for i, t in enumerate(session.texts):
        t = html.escape(t)
        if i < idx:
            parts.append(t)
        elif i == idx:
            parts.append(f"<mark style='background:{SAMPLED};color:black'>{t}</mark>")
        else:
            parts.append(f"<span style='opacity:.3'>{t}</span>")
    return (
        "<div style='white-space:pre-wrap;font-family:monospace;max-width:900px'>"
        + "".join(parts)
        + "</div>"
    )

In [ ]:
session = Session()

tau_w = widgets.FloatSlider(
    value=0.95,
    min=0.05,
    max=0.99,
    step=0.01,
    description="tau",
    continuous_update=False,
)
temp_w = widgets.FloatSlider(
    value=1.0,
    min=0.1,
    max=2.0,
    step=0.05,
    description="temperature",
    continuous_update=False,
)
regen_b = widgets.Button(description="Regenerate", button_style="primary")
next_b = widgets.Button(description="Next token")
step_w = widgets.IntSlider(
    value=0, min=0, max=0, description="step", continuous_update=True
)
text_w = widgets.HTML()
out = widgets.Output()


def redraw(*_):
    if not session.steps:
        return
    idx = min(step_w.value, len(session.steps) - 1)
    text_w.value = render_text(idx)
    with out:
        clear_output(wait=True)
        fig = render_step(session.steps[idx])
        display(fig)
        plt.close(fig)


def on_regenerate(_):
    session.reset()
    for _ in range(MAX_NEW_TOKENS):
        session.advance(tau_w.value, temp_w.value)
    step_w.max = len(session.steps) - 1
    step_w.value = 0
    redraw()


def on_next(_):
    session.advance(tau_w.value, temp_w.value)
    step_w.max = len(session.steps) - 1
    step_w.value = step_w.max
    redraw()


regen_b.on_click(on_regenerate)
next_b.on_click(on_next)
step_w.observe(redraw, "value")

display(
    widgets.VBox(
        [
            widgets.HBox([tau_w, temp_w]),
            widgets.HBox([regen_b, next_b]),
            step_w,
            text_w,
            out,
        ]
    )
)
on_regenerate(None)

### Things to try

* **τ = 0.2 vs 0.95.** A small τ keeps a handful of tokens closest to the entropy. Look at confident steps (low $H$,
  one dominant token): typical sampling still keeps that token because its information is close to the small $H$.
* **High-entropy steps** (e.g. right after a comma): the kept set is wide and the greedy token may be dropped.
* **Temperature** reshapes $p$ *before* the filter, which changes $H$ and therefore *which* tokens count as typical,
  not only how the kept set is weighted.
* Compare the ▼ greedy token with the ★ pick across steps: how often do they differ at τ = 0.95, and at τ = 0.3?
* Change `SEED`, `PROMPT` and `MIN_TOKENS_TO_KEEP` in the constants cell and rerun from there.